# 09 · Explore CellBridge predictions in tissue

A visual workbench for discussing **fine-grained single-cell phenotyping** with biological collaborators. Start with **NuCLS** or **PhenoCell**; HEST sections are optional. All plots call the model **CellBridge** (all training cells, adapted representation, last four transformer blocks adapted, linear classifier). This notebook reads the saved conformal evaluation; it does not train a model or change calibration.

**How to use:** run all cells once, choose a dataset below, and click **Load dataset**. Choose a field, then a patch or a cell, and click **Show selection**. Every view uses the branch-graph conformal method. A later section maps whole HEST sections patch by patch and opens any patch in that same view; the last section projects a HEST patch into its TERRA transcriptomic embeddings. Each section runs one short call; the code behind it lives in `src/cellbridge/qualitative/`.

**Environment:** use the lab's `hest` Python kernel, which contains the image readers and widgets. To start Jupyter from a terminal:
```bash
module load ISG/conda
conda activate hest
cd /lustre/scratch126/cellgen/lotfollahi/so25/projects/cellbridge
PYTHONPATH=src jupyter lab notebooks/09_qualitative_conformal.ipynb
```
The notebook needs this checkout, prepared datasets and saved results on the lab filesystem; it needs neither a GPU nor the sandbox scripts. Paths below respect `CELLBRIDGE_DATA_ROOT` and `CELLBRIDGE_QUALITATIVE_RESULTS` for another machine.

## What am I looking at?

| Term | Meaning |
|---|---|
| Ground truth | The dataset's reference annotation, harmonized to the evaluated cell classes. It is a reference label, not an independent biological verification. The original annotation is also shown. |
| Predicted label | CellBridge's highest-scoring class, shared by all conformal methods. |
| Prediction set | The candidate labels stored by the selected conformal method. An empty set means no label was returned. |
| Lowest common ancestor (LCA) | The most specific common parent of the candidate labels in the Cell Ontology. For example, distinct lymphocyte types can share a broader lymphocyte ancestor. |
| Ancestor-expanded set | All **evaluated classes** under that ancestor. This may add labels absent from the original set; its coverage is reported separately. |
| Depth / coarseness | Number of steps **up from the ground-truth ontology term to the LCA**, along the shortest path. **0 = same term; larger = less specific.** This is the retrospective uncertainty convention in the ancestor-coarseness script, not depth measured down from the ontology root and not a confidence probability. |

A black **×** means the ancestor excludes the reference label; no depth is assigned. An empty set has a red **+**. Cells without a saved test prediction have hollow grey markers and no uncertainty score. These markers represent annotated cell/nucleus centres, not segmentation boundaries or every visible cell.

The ontology is a graph: a type can have more than one parent and sometimes more than one LCA. We list all lowest candidates and choose one deterministically, following the supplied ancestor script: fewest descendant classes, then greatest root depth, then term ID. Synthetic classifier-leaf edges are excluded from the reported depth. A singleton retains its exact classifier class when forming the expanded set.

The saved nominal coverage is a population-level target. Observed coverage in a selected patch can differ substantially; a selected example is not a benchmark estimate. Smoothing, when enabled, shows a local mean among ancestor-covered cells only and cannot repair excluded cells.

In [ ]:
import sys
from pathlib import Path

# Find the checkout when launched from the repository, notebooks/, or workspace.
REPO = next((p for base in (Path.cwd(), *Path.cwd().parents)
             for p in (base, base / "projects/cellbridge")
             if (p / "src/cellbridge/config.py").is_file()), None)
if REPO is None:
    raise FileNotFoundError("Launch Jupyter from the cellbridge checkout.")
sys.path.insert(0, str(REPO / "src"))
from cellbridge.qualitative import views

## Available datasets and sections

Only entries with saved cell-level predictions appear in the menus. HEST remains exploratory; TENX143 is omitted to match notebook 08. The default field has the most evaluated cells (ties use field ID), **not** the best performance. NuCLS has fewer evaluated classes than original annotation types because the saved evaluation filtered classes by support.

In [ ]:
catalog = views.show_catalog()

## Where the code lives

`src/cellbridge/qualitative/` holds everything this notebook runs. `data.py` joins predictions by stable cell ID, checks labels against the dataset mapping and reproduces the saved coverage/set-size metrics before any tissue is shown; `figures.py` draws the views; `views.py` holds the controls. PhenoCell is reconstructed from its original padded tiles; HEST reads only the selected region of its whole-slide image. **After the code is updated, restart the kernel** so the notebook picks up the change.

## Explore

1. **Load dataset** lists its fields and validates its saved predictions. HEST section IDs are in the dataset menu.
2. Choose a **field**, then a **patch**. Grid coordinates are in original H&E pixels, starting at the top left. Only grid patches containing evaluated cells are listed. A PhenoCell field is reconstructed before it can be browsed; the first load from the shared filesystem can take several minutes.
3. Choose **Patch** to see the patch, or **Cell** to centre a neighbourhood on a cell from it. The cell list includes the plot number, stable row ID and original annotation. Enable **Cells from whole field** to find a cell outside the selected patch. **Find cell** filters by row ID or original annotation; suggestions are limited to 500 so large HEST sections remain responsive.
4. Click **Show selection**. **Save displayed view** exports the last rendered figure, complete cell table and provenance to a new folder under `outputs/reproducibility/figures/qualitative_conformal/<your username>/<dataset>/` in the lab workspace; merely changing a control does not change the saved view.

**Method** is fixed to **branch graph**: candidate sets are calibrated within groups of related cell types and are formed along the ontology rather than by thresholding scores alone. The saved evaluation holds seven further methods and this notebook still checks all of their stored metrics when a dataset loads, but only branch graph is offered here, so every view is directly comparable. [Notebook 08](08_conformal_prediction.ipynb) compares the methods against each other.

Optional smoothing uses a 15 µm Gaussian bandwidth, masks unstained tissue, and requires a nearby covered cell (≤30 µm) and weighted support of at least three cells. It is calculated within the displayed crop, so values near its boundary can change when the crop changes. Colours use a fixed scale across methods and fields within a dataset/section; ontology depth is not normalized between datasets.

In [ ]:
explorer = views.Explorer(catalog).show()

## Tissue-level view of a HEST section

The explorer above starts from one patch. This section starts from the whole slide: every grid patch of a HEST section is reduced to one number, painted over the H&E, and can then be opened in the same four-panel detail view.

1. Choose a **section**, a **statistic**, a **patch size** and the minimum number of evaluated cells a patch must hold, then click **Show tissue map**. The left panel is the H&E; the right panel paints each patch by the chosen statistic. **Red is always the worse end.**
2. **Hover any painted patch** to read its name, its pixel bounds, its evaluated-cell count and its value.
3. The **Patch** menu lists the same patches worst first, with the same names. Pick one from the menu and click **Open selected patch**; with the menu focused, typing a name read off the map jumps to it. The patch opens above the notes, with a one-line summary of its cells.
4. **Save displayed view** saves the map and its patch table — and, once a patch is open, that patch and its cell table — to the same per-user folder as the explorer.

| Statistic | Per patch, over its evaluated cells |
|---|---|
| Mean / median depth · coarseness | Steps up from the reference term to the LCA, over **ancestor-covered cells only**. Excluded cells carry no depth and are counted by the next row instead. |
| Mean prediction-set size | Mean number of candidate labels in the selected method's saved set. |
| Reference excluded by ancestor | Fraction of cells whose reference label falls outside the ancestor-expanded set — the black **×** of the detail view. |
| Empty prediction sets | Fraction of cells with an empty saved set — the red **+**. |
| Ancestor-expanded / original-set coverage | Fraction of cells whose reference label is in the expanded set, or in the saved set itself. |
| Top-1 accuracy | Fraction of cells whose highest-scoring label equals the reference. |

Patches use the grid of the patch menu above (`x // size`, `y // size` in level-0 H&E pixels), so a 512 px tissue patch is one of that menu's patches. A patch is painted only when it holds at least the chosen number of evaluated cells; the rest of the tissue is left plain. **Only the held-out test region of a section carries evaluated cells**, so the painted area is usually a part of the slide rather than all of it. Patch size is given in level-0 pixels, whose physical size differs between sections; the figure states the µm equivalent.

Depth keeps the fixed ontology scale of the detail view, so depth maps are comparable between sections, methods and patch sizes. Every other statistic is stretched to the 2nd–98th percentile of the painted patches, which makes its pattern readable but comparable only within that map; its colour-bar label says so. A patch mean over a few dozen cells is noisy — raise **Min cells** before reading a pattern into it, and remember that the saved nominal coverage is a population-level target, not a per-patch guarantee.

The overview reads a low-resolution pyramid level of the whole slide, while the detail view still reads full-resolution pixels. Changing a control does not redraw the map; click **Show tissue map** again. **Open selected patch** renders the chosen patch without redrawing the map, and the TERRA section below projects that same patch.

In [ ]:
tissue_map = views.TissueMap(catalog).show()

## Cross-check a patch against TERRA transcriptomic embeddings

CellBridge reads H&E only. The HEST sections also carry **TERRA** embeddings, saved during preprocessing: for every cell, a 384-dimensional representation of its own Xenium transcript counts (*cell embedding*) and one of its spatial transcriptomic neighbourhood (*neighbourhood embedding*). This section takes the patch currently chosen above and asks one question: **do the cells CellBridge is unsure about also sit in ambiguous places of the transcriptomic space?**

1. Show a tissue map above and pick a **Patch**. This section always uses that patch, that section and the same method.
2. Choose the **Embedding**, what to **Colour by**, and how many **Context** cells to project alongside the patch. Click **Fetch TERRA embeddings**. Nothing is computed until you click; 20,000 context cells take a minute or two and 50,000 several minutes, because the projection is seeded for reproducibility and therefore single-threaded.
3. Cells with no saved test prediction stay as hollow grey markers, as everywhere else in this notebook; the black **×** and red **+** keep their meanings.
4. **Save displayed view** saves the figure, the patch's cells with their UMAP coordinates and neighbour agreement, and the settings, to the same per-user folder.

| Panel | Reads as |
|---|---|
| UMAP | PCA to 50 components, then UMAP (`n_neighbors=15`, `min_dist=0.1`, `random_state=0`) over the patch cells plus a uniform random sample of the section's other cells. The patch is in colour, the context in light grey. Axes have no units; only neighbourhoods mean anything, and distances between separated groups do not. |
| The same patch in tissue | The same cells and colours over the H&E crop, so a group seen in the UMAP can be located in the tissue. |
| Neighbourhood agreement | For each evaluated patch cell, the share of its 15 nearest cells **in TERRA PCA space** carrying the same reference label, averaged within each depth level. Black is the excluded group, red the empty sets, and `n` is the number of cells behind each bar. |

**Read this as support, not as validation.** The HEST reference labels are themselves derived from the transcript counts TERRA embeds, so high agreement at depth 0 is expected and is not independent evidence that a label is right. What is informative is the contrast: whether CellBridge's H&E-derived uncertainty — larger depth, an excluded reference, an empty set — falls on cells whose transcriptomic neighbourhood is also mixed. Where it does, the coarser prediction reflects ambiguity that is present in the tissue; where agreement stays high while depth grows, the uncertainty is a limitation of the H&E view rather than of the cell.

Context cells are drawn uniformly from the whole section, including cells with no test prediction, so the grey backdrop shows the section's transcriptomic landscape rather than only its evaluated part. Agreement is measured among the evaluated cells of the projected sample, so a larger context makes it less dependent on the patch's own composition. UMAP is fitted afresh on every click: the layout changes when the embedding or the context size changes, and only structure within one figure should be compared. A patch of a few dozen cells supports no claim about the section.

In [ ]:
terra = views.TerraView(tissue_map).show()

## Notes for your biological review

For a selected cell, compare the reference annotation with its morphology and neighbouring cells. Does the top prediction look plausible? If it is wrong, does the original candidate set still contain the reference? Does the common ancestor retain a useful biological identity, or become too broad? Inspect excluded cells alongside covered cells with large depth; the two represent different limitations. For a HEST patch, the TERRA section shows whether an uncertain cell is also ambiguous transcriptomically before you conclude that CellBridge is simply wrong.

For reproducibility, save the displayed view and record why you chose it. A favourable HEST example should be labelled as such if later selected by performance. HEST reference labels come from the prepared molecular annotations and are aligned to H&E; neighbouring cells without saved test predictions remain visible but are not scored. The displayed context crop is for review and need not equal the model's input crop.

**Technical references within this project:** [notebook 08](08_conformal_prediction.ipynb), the pinned mappings in `configs/preprocess/cell_ontology/`, `cellbridge.inference.conformal.graph`, and this notebook's own code in `src/cellbridge/qualitative/`. The LCA and spatial depth conventions follow the supplied `plot_ancestor_coarseness.py` and `plot_phenocell_uncertainty.py`; no code is imported from those scratch scripts. The notebook recomputes the LCA of each saved set, which can differ from a graph method's stored anchor when several ontology nodes have the same descendant classes.